# Введение

Это pet-project по разработке базы данных для компании, занимающейся покупкой, ремонтом и продажей подержанных автомобилей.

## Цель проекта
Создать структурированную базу данных для хранения и обработки информации об автомобилях, клиентах, сделках и сотрудниках компании.

##  Основные задачи системы
- Хранение данных об автомобилях (покупка, состояние, история)  
- Учет ремонтов и затрат на СТО  
- Фиксация сделок покупки и продажи  
- Ведение базы клиентов и их заявок  
- Контроль работы менеджеров  

## Анализируемые показатели
- Прибыль по каждому автомобилю  
- Время оборота автомобиля (от покупки до продажи)  
- Затраты на ремонт  
- Эффективность менеджеров  
- Конверсия заявок в сделки  


# ***Структура базы данных***

В рамках проекта была спроектирована реляционная база данных, отражающая основные бизнес-процессы компании.

## ***Основные сущности***

### *cars*
Таблица с информацией об автомобилях.

- id — уникальный идентификатор  
- brand — марка автомобиля  
- model — модель  
- year — год выпуска  
- mileage — пробег  
- status — текущий статус (available / sold / repair)  

### *clients*
Таблица с данными клиентов.

- id — уникальный идентификатор  
- name — имя клиента  
- phone — контактный телефон  

### *employees*
Таблица сотрудников компании.

- id — уникальный идентификатор  
- name — имя сотрудника  
- position — должность  

### *deals*
Таблица сделок покупки и продажи.

- id — уникальный идентификатор  
- car_id — ссылка на автомобиль  
- client_id — ссылка на клиента  
- employee_id — менеджер сделки  
- deal_type — тип сделки (purchase / sale)  
- price — стоимость  
- deal_date — дата сделки  

### *repairs*
Таблица ремонтов автомобилей.

- id — уникальный идентификатор  
- car_id — ссылка на автомобиль  
- description — описание работ  
- cost — стоимость ремонта  
- repair_date — дата  

### *requests*
Таблица заявок клиентов.

- id — уникальный идентификатор  
- client_id — ссылка на клиента  
- car_id — интересующий автомобиль  
- status — статус заявки  

# ***Создание таблиц***


In [ ]:
CREATE TABLE cars (
    id SERIAL PRIMARY KEY,
    brand VARCHAR(50),
    model VARCHAR(50),
    year INT,
    mileage INT,
    status VARCHAR(20)
);

CREATE TABLE clients (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100),
    phone VARCHAR(20)
);

CREATE TABLE employees (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100),
    position VARCHAR(50)
);

CREATE TABLE deals (
    id SERIAL PRIMARY KEY,
    car_id INT,
    client_id INT,
    employee_id INT,
    deal_type VARCHAR(20),
    price NUMERIC(10, 2),
    deal_date DATE,
    FOREIGN KEY (car_id) REFERENCES cars(id),
    FOREIGN KEY (client_id) REFERENCES clients(id),
    FOREIGN KEY (employee_id) REFERENCES employees(id)
);

CREATE TABLE repairs (
    id SERIAL PRIMARY KEY,
    car_id INT,
    description TEXT,
    cost NUMERIC(10, 2),
    repair_date DATE,
    FOREIGN KEY (car_id) REFERENCES cars(id)
);

CREATE TABLE requests (
    id SERIAL PRIMARY KEY,
    client_id INT,
    car_id INT,
    status VARCHAR(20),
    FOREIGN KEY (client_id) REFERENCES clients(id),
    FOREIGN KEY (car_id) REFERENCES cars(id)
);

##На данном этапе:
- Созданы все основные таблицы

- Заданы первичные ключи (PRIMARY KEY)

- Настроены внешние ключи (FOREIGN KEY) для связи данных

- Определены типы данных для каждого поля

#**Заполнение данных**

In [ ]:

INSERT INTO cars (brand, model, year, mileage, status) VALUES
('Toyota', 'Camry', 2018, 90000, 'available'),
('BMW', 'X5', 2017, 120000, 'repair'),
('Audi', 'A4', 2019, 70000, 'sold');

INSERT INTO clients (name, phone) VALUES
('Иван Иванов', '+79991234567'),
('Петр Петров', '+79997654321');

INSERT INTO employees (name, position) VALUES
('Анна Смирнова', 'manager'),
('Олег Кузнецов', 'manager');

INSERT INTO deals (car_id, client_id, employee_id, deal_type, price, deal_date) VALUES
(1, 1, 1, 'purchase', 800000, '2023-01-10'),
(3, 2, 2, 'sale', 1200000, '2023-02-15');

INSERT INTO repairs (car_id, description, cost, repair_date) VALUES
(2, 'Замена двигателя', 150000, '2023-01-20');

INSERT INTO requests (client_id, car_id, status) VALUES
(1, 1, 'new'),
(2, 3, 'completed');

##На данном этапе:
- Добавлены тестовые записи во все таблицы

- Обеспечена связность данных между таблицами

- Подготовлена база для выполнения аналитических запросов


# **Изменение данных (UPDATE / DELETE)**

## *Обновление данных*

In [ ]:
UPDATE cars
SET status = 'sold'
WHERE id = 1;

## *Удаление данных*

In [ ]:
DELETE FROM requests
WHERE status = 'completed';

#**Аналитические запросы**

###*Прибыль по каждому автомобилю*

In [ ]:
SELECT
    c.id,
    c.brand,
    c.model,
    SUM(CASE WHEN d.deal_type = 'sale' THEN d.price ELSE 0 END) -
    SUM(CASE WHEN d.deal_type = 'purchase' THEN d.price ELSE 0 END) AS profit
FROM cars c
JOIN deals d ON c.id = d.car_id
GROUP BY c.id, c.brand, c.model;

###*Средние затраты на ремонт по автомобилям*

In [ ]:
SELECT
    c.brand,
    c.model,
    AVG(r.cost) AS avg_repair_cost
FROM cars c
JOIN repairs r ON c.id = r.car_id
GROUP BY c.brand, c.model;

###*Топ менеджеров по сумме продаж*

In [ ]:
SELECT
    e.name,
    SUM(d.price) AS total_sales
FROM employees e
JOIN deals d ON e.id = d.employee_id
WHERE d.deal_type = 'sale'
GROUP BY e.name
ORDER BY total_sales DESC;

###*Количество заявок по статусам*

In [ ]:
SELECT
    status,
    COUNT(*) AS total_requests
FROM requests
GROUP BY status;

# ***Заключение***

В рамках данного проекта была спроектирована и реализована реляционная база данных для компании, занимающейся продажей подержанных автомобилей.

Разработанная база данных позволяет:
- Хранить и структурировать информацию об автомобилях, клиентах и сделках  
- Анализировать прибыль и затраты  
- Оценивать эффективность работы сотрудников  
- Отслеживать бизнес-процессы компании  

Таким образом, поставленная цель проекта была достигнута, а созданная система может быть использована как основа для дальнейшего развития и масштабирования.